# SLM-RL — The Full Journey (Code Edition)

This notebook mirrors the web UI's onboarding flow, step by step, but everything happens in code.

**The loop — four moves, one generation at a time:**

| Step | What happens |
|------|-------------|
| **01 — Play** | Your model rolls out episodes in a text-native Atari game — Boxing, Space Invaders, and more. |
| **02 — Train** | We turn those episodes into a dataset and fine-tune. Better play only sticks if eval says so. |
| **03 — Compare** | Theater shows base vs champion on the same seeds — the workshop money shot. |
| **04 — Publish** | Push datasets and adapters to your Hugging Face account when you're ready to share. |

**Runtime → Change runtime type → GPU (T4 or better)** for any real training.

---

## Step 0 — Install

Clone the repo and install dependencies. The hardware tier is auto-detected — no manual choices needed.

In [ ]:
# @title Clone + install SLM-RL
REPO_URL = "https://github.com/craftsmanlabs/SLM-RL.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/content/SLM-RL")
here = Path.cwd()
if (here / "pyproject.toml").is_file() and (here / "slm_rl").is_dir():
    ROOT = here
elif (here / "SLM-RL" / "pyproject.toml").is_file():
    ROOT = here / "SLM-RL"
elif not (ROOT / "pyproject.toml").is_file():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {ROOT}
else:
    print(f"already cloned: {ROOT}")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-e", ".[atari]")
pip("torch", "ipywidgets", "matplotlib", "huggingface_hub")

import torch
print("cwd:", Path.cwd())
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## Step 1 — Welcome & credentials

In the web UI, this is the landing page where you enter your name and (optionally) a Hugging Face token.
Here we just set the same variables — the token only matters if you want to publish at the end.

In [ ]:
# @title Credentials (mirrors the Welcome screen)
DISPLAY_NAME = "Ada"  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

assert DISPLAY_NAME.strip(), "Display name is required, even to skip the token."
if HF_TOKEN.strip() and not HF_TOKEN.strip().startswith("hf_"):
    raise ValueError("Paste a Hugging Face token starting with hf_.")

print(f"Welcome, {DISPLAY_NAME}!")
print(f"HF token: {'set (' + HF_TOKEN[:6] + '…)' if HF_TOKEN.strip() else 'skipped — publish disabled'}")
print("\nYou're onboarded. Let's detect your hardware.")

## Step 2 — Detect hardware & inspect your tier

The platform auto-detects your machine and picks a model, backend, and training strategy.
This is what `slm-rl info` does on the CLI.

In [ ]:
# @title Hardware detection (mirrors `slm-rl info`)
from slm_rl.config.loader import load_tiers
from slm_rl.games.registry import available_games
from slm_rl.platform.hardware import detect_host, resolve_tier

host = detect_host()
tier = resolve_tier(load_tiers(), host)

print(f"Host:  os={host.os}  ram={host.ram_gb:.1f}GB  "
      f"cuda_vram={host.cuda_vram_gb and f'{host.cuda_vram_gb:.1f}GB'}  mps={host.has_mps}")
print(f"Tier:  {tier.name}")
print(f"Model: {tier.model}")
print(f"Backend: {tier.backend}  |  Training: {tier.train}")
print(f"Games: {', '.join(available_games())}")

## Step 3 — Create an experiment

In the web UI, this is the **New Project** modal.
You pick a game, set training knobs, and optionally point at a pre-baked dataset pack.
Here we configure the same things as Python variables, then materialize the run config.

In [ ]:
# @title Experiment config (mirrors the New Project modal)
from slm_rl.games.registry import available_games

# --- Pick your game ---
GAME = "boxing"  # @param ["boxing", "space-invaders", "freeway", "demon-attack"]
assert GAME in available_games(), f"{GAME} not registered. Available: {available_games()}"

# --- Workshop pack URLs (pre-filled like the UI) ---
WORKSHOP_ORG = "BLANK"
DATASET_URL = f"{WORKSHOP_ORG}/slm-rl-{GAME}"  # @param {type:"string"}
ADAPTER_URL = f"{WORKSHOP_ORG}/slm-rl-{GAME}"  # @param {type:"string"}
DQN_URL = f"{WORKSHOP_ORG}/slm-rl-{GAME}-dqn"  # @param {type:"string"}

# --- Training knobs (workshop defaults from the UI) ---
GENERATIONS = 2  # @param {type:"integer"}
EPISODES_PER_GENERATION = 2  # @param {type:"integer"}
EVAL_EPISODES = 2  # @param {type:"integer"}
MAX_TURNS = 30  # @param {type:"integer"}
SELECTION_QUANTILE = 0.25  # @param {type:"number"}
TRAIN_STRATEGY = "grpo"  # @param ["grpo", "reject_sft"]
GRPO_MAX_STEPS = 24  # @param {type:"integer"}
GRPO_MAX_PROMPTS = 32  # @param {type:"integer"}

# --- Run identity ---
import re
RUN_NAME = f"colab-{DISPLAY_NAME.lower().replace(' ', '-')}"  # @param {type:"string"}
RUN_NAME = re.sub(r"[^a-z0-9-]", "", RUN_NAME)[:40]
RUN_ID = f"pg-{RUN_NAME}"

print(f"Experiment: {RUN_NAME}")
print(f"  game:       {GAME}")
print(f"  run_id:     {RUN_ID}")
print(f"  model:      {tier.model}")
print(f"  dataset:    {DATASET_URL or '(none — live warm-start)'}")
print(f"  adapter:    {ADAPTER_URL or '(none)'}")
print(f"  strategy:   {TRAIN_STRATEGY}")
print(f"  gens:       {GENERATIONS}  |  eps/gen: {EPISODES_PER_GENERATION}")

In [ ]:
# @title Materialize run config
from pathlib import Path
from slm_rl.config.loader import load_run_config, load_game_config

HOME = "/content/slm-rl-runs"
Path(HOME).mkdir(parents=True, exist_ok=True)

overrides = {
    "run_id": RUN_ID,
    "home": HOME,
    "model": tier.model,
    "backend": tier.backend,
    "tier": tier.name,
    "train_strategy": TRAIN_STRATEGY,
    "dataset_url": DATASET_URL or None,
    "dqn_url": DQN_URL or None,
    "adapter_url": ADAPTER_URL or None,
    "train": {
        "episodes_per_generation": EPISODES_PER_GENERATION,
        "selection_quantile": SELECTION_QUANTILE,
        "grpo_max_steps": GRPO_MAX_STEPS,
        "grpo_max_prompts": GRPO_MAX_PROMPTS,
        "max_completion_tokens": 24,
    },
}

run_cfg = load_run_config(game=GAME, overrides=overrides)
game_cfg = load_game_config(GAME)

print(f"Run config materialized:")
print(f"  home:     {run_cfg.home}")
print(f"  run_id:   {run_cfg.run_id}")
print(f"  game:     {run_cfg.game}")
print(f"  model:    {getattr(run_cfg, 'model', tier.model)}")

## Step 4 — Evolve: the self-improvement loop

This is the heart of the workshop. In the web UI you click **Evolve** in the project workspace.
Here we run `GenerationRunner` directly — the same code the CLI `slm-rl evolve` uses.

The loop for each generation:
1. **Rollout** — the champion (or base model) plays episodes
2. **Dataset** — consolidate rollouts, select top quantile
3. **Train** — GRPO or reject-SFT fine-tuning
4. **Eval** — frozen eval suite (no training aids)
5. **Gate** — promote only if the candidate beats the champion

In [ ]:
# @title Run evolve loop
from slm_rl.orchestrator.generation import GenerationRunner
from slm_rl.orchestrator.registry import ModelRegistry
from slm_rl.packs import import_adapter_as_champion, resolve_adapter, resolve_pack, materialize_rollouts, resolve_dqn

runner = GenerationRunner(run_cfg)
print(f"[evolve] run_id={run_cfg.run_id}  game={run_cfg.game}  "
      f"backend={runner.backend_name}  model={runner.model_id}")

# --- Gen 0: baseline (raw model, no training) ---
print("\n--- Gen 0: Baseline eval (raw model) ---")
runner.ensure_baseline(skip=False)
start = runner.registry.next_generation
end = start + GENERATIONS
print(f"Baseline done. Next generation: {start}")

# --- Gen 1: import SFT adapter or baked pack or live warm-start ---
sft_url = (ADAPTER_URL or "").strip() or None
pack_url = (DATASET_URL or "").strip() or None

if sft_url and start == 1:
    print(f"\n--- Gen 1: Importing published SFT adapter from {sft_url} ---")
    src = resolve_adapter(sft_url, run_cfg.home, run_cfg.game)
    import_adapter_as_champion(
        runner.paths.root, src, model_id=runner.model_id, game=run_cfg.game,
    )
    runner.registry = ModelRegistry(runner.paths.registry)
    print(f"Gen 1 imported. Champion: {runner.registry.champion}")
    start = runner.registry.next_generation
    end = start + GENERATIONS
elif pack_url and start == 1:
    print(f"\n--- Gen 1: Warm-start from baked pack {pack_url} ---")
    pack = resolve_pack(pack_url, run_cfg.home, run_cfg.game)
    materialize_rollouts(pack, runner.paths.rollouts(1))
    dqn = (DQN_URL or "").strip() or None
    if dqn:
        pt = resolve_dqn(dqn, run_cfg.home, run_cfg.game)
        run_cfg.teacher.dqn_checkpoint = str(pt)
    m = runner.run_generation(1, teacher=True, skip_rollout=True, skip_eval=False)
    print(f"Gen 1: primary={m['eval'].get('primary', float('nan')):.3f}  "
          f"promoted={m['gate']['promoted']}  ({m['gate']['reason']})")
    start = runner.registry.next_generation

# --- Gen 2+: RL self-improvement loop ---
print(f"\n--- RL loop: gen {start} to {end - 1} ---")
stop_after = runner.cfg.gate.max_consecutive_failures

for g in range(start, end):
    print(f"\n  Generation {g}:")
    m = runner.run_generation(g)
    promoted = m['gate']['promoted']
    primary = m['eval']['primary']
    reason = m['gate']['reason']
    print(f"    primary={primary:.3f}  promoted={promoted}  ({reason})")

    fails = runner.registry.consecutive_failures
    if not promoted and fails >= stop_after:
        print(f"    Early stop: {fails} consecutive rejects (limit {stop_after})")
        break

print(f"\nEvolve complete. Champion: {runner.registry.champion}")

## Step 5 — Theater: base vs champion

In the web UI this is the **Theater** button. Same seeds, same game — base model vs your trained champion side by side.
Exhibition seeds start at 20,000 (disjoint from training and eval seeds).

In [ ]:
# @title Run theater exhibition
from slm_rl.theater.exhibition import run_exhibition
from slm_rl.orchestrator.paths import RunPaths

THEATER_EPISODES = 5  # @param {type:"integer"}

paths = RunPaths(HOME, RUN_ID)

result = run_exhibition(
    paths.root, GAME,
    episodes=THEATER_EPISODES,
    seed_start=20_000,
)

print(f"Base exhibition:      {result.base_dir}")
if result.champion_dir is None:
    print(f"No champion yet: {result.message}")
else:
    print(f"Champion (gen {result.champion_generation}): {result.champion_dir}")

In [ ]:
# @title Visualize theater results
import json
from pathlib import Path

def summarize_side(side_dir: Path, label: str):
    """Print a summary of theater episodes from JSONL files."""
    jsonl_files = sorted(side_dir.glob("*.jsonl"))
    if not jsonl_files:
        print(f"  {label}: no episodes found")
        return

    episodes = {}
    for jf in jsonl_files:
        with jf.open() as f:
            for line in f:
                rec = json.loads(line)
                eid = rec.get("episode_id", "?")
                if eid not in episodes:
                    episodes[eid] = {"steps": 0, "reward": 0.0, "outcome": "?"}
                episodes[eid]["steps"] += 1
                episodes[eid]["reward"] += rec.get("reward", 0.0)
                if rec.get("outcome"):
                    episodes[eid]["outcome"] = rec["outcome"]

    print(f"\n  {label} ({len(episodes)} episodes):")
    print(f"  {'Episode':<30} {'Steps':>6} {'Reward':>8} {'Outcome':>10}")
    print(f"  {'─' * 56}")
    for eid, info in episodes.items():
        print(f"  {eid:<30} {info['steps']:>6} {info['reward']:>8.1f} {info['outcome']:>10}")


theater_dir = paths.root / "theater"
base_dir = theater_dir / "base"
champ_dir = theater_dir / "champion"

print("=" * 60)
print("THEATER RESULTS — Base vs Champion")
print("=" * 60)

if base_dir.exists():
    summarize_side(base_dir, "BASE (raw model)")
else:
    print("  Base: not found")

if champ_dir.exists():
    summarize_side(champ_dir, "CHAMPION (trained)")
else:
    print("  Champion: not found (no generation was promoted)")

## Step 6 — Inspect generations

Browse the generation lineage — each generation's metrics, gate decisions, and adapter paths.

In [ ]:
# @title Generation summary
import json
from pathlib import Path

gens_dir = paths.root / "generations"
gen_dirs = sorted(gens_dir.glob("gen_*")) if gens_dir.exists() else []

print(f"Run: {RUN_ID}  |  Game: {GAME}  |  {len(gen_dirs)} generation(s)")
print("=" * 72)

for gd in gen_dirs:
    gen_num = gd.name
    metrics_path = gd / "metrics.json"
    adapter_dir = gd / "adapter"

    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        eval_m = metrics.get("eval", {})
        gate_m = metrics.get("gate", {})
        primary = eval_m.get("primary", "n/a")
        promoted = gate_m.get("promoted", "n/a")
        reason = gate_m.get("reason", "")
        print(f"\n{gen_num}:")
        print(f"  primary:  {primary}")
        print(f"  promoted: {promoted}  ({reason})")
        print(f"  adapter:  {'yes' if adapter_dir.exists() else 'no'}")
        if "rollout" in metrics:
            r = metrics["rollout"]
            print(f"  rollout:  {r.get('episodes', '?')} eps, "
                  f"win_rate={r.get('win_rate', '?')}")
    else:
        print(f"\n{gen_num}: (no metrics yet)")

# Registry snapshot
reg_path = paths.root / "registry.json"
if reg_path.exists():
    reg = json.loads(reg_path.read_text())
    print(f"\nChampion: {reg.get('champion', 'none')}")
    print(f"Next generation: {reg.get('next_generation', '?')}")

## Step 7 — Publish to Hugging Face

In the web UI this is the **Publish** button. Push your datasets and champion adapter to your HF account.
Requires a write token (set in Step 1). Skip this cell if you didn't provide one.

In [ ]:
# @title Publish datasets + adapter
PUBLISH_REPO = ""  # @param {type:"string"}

if not HF_TOKEN.strip():
    print("No HF token set — skipping publish.")
    print("Set HF_TOKEN in Step 1 and re-run to enable.")
elif not PUBLISH_REPO.strip():
    print("Set PUBLISH_REPO above (e.g. 'your-username/slm-rl-boxing') and re-run.")
else:
    import os
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

    from slm_rl.datagen.hf_push import push_generation

    gens = sorted(paths.root.glob("generations/gen_*"))
    if not gens:
        print("No generations to push.")
    else:
        for gen_dir in gens:
            g = int(gen_dir.name.split("_")[1])
            try:
                url = push_generation(PUBLISH_REPO, RUN_ID, g, gen_dir, private=True)
                print(f"gen {g} -> {url}")
            except Exception as e:
                print(f"gen {g}: push failed — {e}")

        # Push champion adapter if it exists
        champion_gen = None
        if reg_path.exists():
            reg = json.loads(reg_path.read_text())
            champion_gen = reg.get("champion")
        if champion_gen is not None:
            adapter = paths.root / f"generations/gen_{champion_gen:03d}/adapter"
            if adapter.exists():
                from huggingface_hub import HfApi
                api = HfApi(token=HF_TOKEN.strip())
                model_repo = PUBLISH_REPO.replace("-data", "")
                try:
                    api.create_repo(model_repo, private=True, exist_ok=True)
                    api.upload_folder(
                        folder_path=str(adapter),
                        repo_id=model_repo,
                        path_in_repo="adapter",
                    )
                    print(f"Champion adapter (gen {champion_gen}) -> https://huggingface.co/{model_repo}")
                except Exception as e:
                    print(f"Adapter push failed: {e}")
            else:
                print(f"No adapter found for champion gen {champion_gen}")
        else:
            print("No champion promoted yet — adapter push skipped.")

## Step 8 — Download run (optional)

Zip the entire run directory for local use or to continue evolving on your own machine.

In [ ]:
# @title Download run directory
import shutil
from pathlib import Path

run_dir = paths.root
assert run_dir.is_dir(), f"Run directory not found: {run_dir}"

zip_path = Path("/content") / f"slm-rl-{RUN_NAME}"
archive = shutil.make_archive(str(zip_path), "zip", root_dir=run_dir)
print(f"Archive: {archive}  ({Path(archive).stat().st_size / 1024 / 1024:.1f} MB)")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not in Colab — download the file manually from the path above.")